# OpenVT Monolayer Benchmark Reference Model

This notebook implements the OpenVT reference model for a growing 2D cell monolayer with contact inhibition using `CorePotts` and `PottsToolkit` on the GPU.

The model is defined by:
- A growing monolayer starting from a single cell.
- Area growth rate $\alpha$ which pauses upon contact inhibition (Type 1: mechanical compression, Type 2: crowding).
- Cell division with stochastic threshold $X_i \sim \mathcal{N}(2, 0.4^2)$.
- No cell-cell adhesion, only steric repulsion.

Before running the main 10,000-cell simulation, we first calibrate the timescale $T$ mechanically using a 1D chain of 11 compressed cells as specified in the OpenVT manuscript.

In [ ]:
using Pkg; Pkg.activate(".")
Pkg.add("CommonSolve")
Pkg.add("StructArrays")
Pkg.add("SciMLBase")
Pkg.add("CSV")
Pkg.add("DataFrames")
Pkg.add("Glob")
Pkg.add("Colors")

Pkg.resolve()
Pkg.instantiate()
Pkg.precompile()
using Metal # Or CUDA, AMDGPU depending on your hardware
backend = MetalBackend()
ArrayType = MtlArray

using SciMLBase
using CommonSolve
using LinearAlgebra
using Statistics
using Random 
using StructArrays
using CorePotts
using CorePotts.KernelAbstractions
using CorePotts.Atomix
using PottsToolkit

## 1. Mechanical Calibration ($T_{relax}$)

The OpenVT reference model dictates that the temporal unit $T$ is the time it takes for an 11-cell compressed chain to relax such that the outer cells are 90% of their fully relaxed distance. The cell cycle duration is defined as $5T$.

We use SciML's `EnsembleProblem` to run this calibration 10 trajectories to get the expected $\langle T_{relax} \rangle$, which is then used to set the growth rate $\alpha$ for the main simulation.

In [ ]:
# Configuration & Initial Compressed Grid
R = 5.0f0
initial_area = Float32(pi * R^2)
relaxed_distance = 10.0f0 * (2.0f0 * R) 
target_distance = 0.90f0 * relaxed_distance

grid_size = (150, 5)

medium = CellType(:Medium, is_background=true)
tissue = CellType(:Tissue)

struct CalibrationLayout <: PottsToolkit.Layouts.AbstractLayout
    tissue_type::CellType
    R::Float32
end

function PottsToolkit.Layouts.build_layout!(layout::CalibrationLayout, ctx::PottsToolkit.Layouts.LayoutContext)
    grid_size = size(ctx.grid)
    type_id = ctx.type_to_id[layout.tissue_type]
    
    for i in 1:11
        cx = 51 + (i - 1) * 5
        cell_id = ctx.next_cell_id
        ctx.next_cell_id += 1
        
        for x in 1:grid_size[1], y in 1:grid_size[2]
            if x >= cx && x < cx + 5
                ctx.grid[x, y] = cell_id
            end
        end
        ctx.cell_type_map[cell_id] = type_id
    end
end

# Define the Calibration PottsSystem (No Growth/Mitosis)
calib_sys = PottsSystem(
    cell_types = [medium, tissue],
    penalties = [
        HSTVolumeComponent(tissue => (λ = 5.0f0, target = 50.0f0);eta = 1.0f0),
        SurfaceAreaComponent(tissue => (λ = 1.0f0, target = 12.0f0*sqrt(initial_area))),
        AdhesionComponent((medium, tissue) => 10.0f0, (tissue, tissue) => 20.0f0),
        # ConnectivityConstraint()
    ]
)

layout_calib = CalibrationLayout(tissue, R)
base_prob = PottsProblem(calib_sys, layout_calib, grid_size; tspan=(0, 50000), topology=CorePotts.MooreTopology{2}())

# Callback to halt when relaxed
function compute_com_x(grid_cpu, cell_id)
    x_coords = [x for x in 1:size(grid_cpu, 1), y in 1:size(grid_cpu, 2) if grid_cpu[x,y] == cell_id]
    return isempty(x_coords) ? 0.0 : mean(x_coords)
end

distance_condition = (u, t, integrator) -> begin
    # Sync grid to CPU briefly to check positions
    grid_cpu = Array(u.grid) 
    com1 = compute_com_x(grid_cpu, 1)
    com11 = compute_com_x(grid_cpu, 11)
    
    # Handle periodic wrapping across 150 boundary for COM calculation
    if com11 < com1
        com11 += 150
    end
    
    return abs(com11 - com1) >= target_distance
end

halt_cb = DiscreteCallback(distance_condition, terminate!)

# Ensemble Execution
alg = SequentialMetropolis(T=20.0f0)
ensemble_prob = EnsembleProblem(base_prob)

println("Running 10 calibration trajectories...")
sol_ensemble = solve(ensemble_prob, alg, EnsembleSerial(); 
                     trajectories = 10, 
                     callback = halt_cb,
                     save_everystep = false)

# Extract T and compute Growth Rate
stopping_times = [sol.t[end] for sol in sol_ensemble.u]
T_relax = mean(stopping_times)
println("Calibration Complete. Expected T_relax = $T_relax steps.")

T_cycle = 5.0f0 * Float32(T_relax)
calibrated_alpha = initial_area / T_cycle
println("Calibrated alpha for monolayer growth: $calibrated_alpha")


In [ ]:
using CairoMakie
CairoMakie.activate!()
using MakiePotts

println("Running and recording a single calibration trajectory...")
viz_sol = solve(base_prob, alg; callback = halt_cb, saveat = 1)

record_potts("calibration_viz.mp4", viz_sol;
    framerate = 15,
    resolution = (1200, 200),
    cell_colors = ["#00E5FF" for _ in 1:11],
    draw_boundaries = true,
    boundary_color = "#FFFFFF"
)
display("Saved video to calibration_viz.mp4")


In [ ]:
struct MonolayerPenalty <: CorePotts.AbstractPenalty{CorePotts.Rigid} end
CorePotts.evaluate_penalty(::MonolayerPenalty, ctx) = 0.0f0

struct MonolayerGrowthComponent <: AbstractComponent
    initial_area::Float32
    initial_div_thresh::Int32
end

import PottsToolkit.Problem: compile_component
import PottsToolkit.System: required_variables

function required_variables(::MonolayerGrowthComponent)
    (
        free_surfaces = Int32,
        inhibition_states = UInt8,
        target_volumes_float = Float32,
        division_threshold_volumes = Int32
    )
end

function compile_component(
        comp::MonolayerGrowthComponent, sys::PottsToolkit.System.PottsSystem,
        type_to_id::Dict{CellType, UInt8}, num_types::Int64)
    trackers = [CorePotts.SurfaceAreaTracker()]

    props = Dict{UInt8, Dict{Symbol, Any}}()
    for (ct, id) in type_to_id
        if ct.name == :Tissue
            props[id] = Dict(
                :target_volumes_float => comp.initial_area,
                :target_volumes => floor(Int32, comp.initial_area),
                :target_surface_areas => floor(Int32, 4.0f0 * sqrt(comp.initial_area)),
                :division_threshold_volumes => comp.initial_div_thresh
            )
        end
    end

    return (MonolayerPenalty(), trackers, props)
end


@kernel function reset_free_surfaces_kernel!(free_surfaces)
    i = @index(Global, Linear)
    free_surfaces[i] = Int32(0)
end

@kernel function accumulate_free_surfaces_kernel!(grid, free_surfaces, grid_dims, topology)
    i = @index(Global, Linear)
    cell_id = grid[i]
    if cell_id > 0
        coords = CorePotts.idx_to_coord(UInt32(i), grid_dims)
        n_med = Int32(0)
        for d in 1:length(CorePotts.offsets(topology))
            n_idx = CorePotts.get_neighbor_by_coord(topology, coords, UInt32(d), grid_dims)
            if grid[n_idx] == 0
                n_med += Int32(1)
            end
        end
        if n_med > 0
            CorePotts.Atomix.@atomic free_surfaces[cell_id] += n_med
        end
    end
end

@kernel function monolayer_growth_kernel!(volumes, target_volumes, target_volumes_float, target_surface_areas, surface_areas, free_surfaces, inhibition_states, N_cells, evt_alpha, evt_beta, evt_gamma, evt_dt)
    i = @index(Global, Linear)
    if i <= N_cells[1]
        v = volumes[i]
        if v > 0
            tv = target_volumes[i]
            inhibited_type1 = (Float32(v) / Float32(tv)) < evt_beta

            fi = surface_areas[i] > 0 ?
                 (Float32(free_surfaces[i]) / Float32(surface_areas[i])) : 0.0f0
            inhibited_type2 = fi < evt_gamma

            state = UInt8(0)
            if inhibited_type1
                state += UInt8(1)
            end
            if inhibited_type2
                state += UInt8(2)
            end
            inhibition_states[i] = state

            if state == 0
                target_volumes_float[i] += evt_alpha * evt_dt
                target_volumes[i] = floor(Int32, target_volumes_float[i])
                target_surface_areas[i] = floor(
                    Int32, 4.0f0 * sqrt(target_volumes_float[i]))
            end
        else
            inhibition_states[i] = UInt8(0)
        end
    end
end

struct ResetFreeSurfacesEvent <: CorePotts.AbstractEvent end
CorePotts.get_event_kernel(::ResetFreeSurfacesEvent, backend, block_size) = reset_free_surfaces_kernel!(backend, block_size)
CorePotts.get_event_args(::ResetFreeSurfacesEvent, mask, u, p, cache, t) = (u.cell_data.free_surfaces,)
CorePotts.get_event_ndrange(::ResetFreeSurfacesEvent, mask, u) = length(u.cell_data.free_surfaces)

struct AccumulateFreeSurfacesEvent <: CorePotts.AbstractEvent end
CorePotts.get_event_kernel(::AccumulateFreeSurfacesEvent, backend, block_size) = accumulate_free_surfaces_kernel!(backend, block_size)
CorePotts.get_event_args(::AccumulateFreeSurfacesEvent, mask, u, p, cache, t) = (u.grid, u.cell_data.free_surfaces, cache.grid_dims, p.topology)
CorePotts.get_event_ndrange(::AccumulateFreeSurfacesEvent, mask, u) = length(u.grid)

struct ApplyMonolayerGrowthEvent <: CorePotts.AbstractEvent
    alpha::Float32
    beta::Float32
    gamma::Float32
    dt::Float32
end
CorePotts.get_event_kernel(::ApplyMonolayerGrowthEvent, backend, block_size) = monolayer_growth_kernel!(backend, block_size)
CorePotts.get_event_args(evt::ApplyMonolayerGrowthEvent, mask, u, p, cache, t) = (u.cell_data.volumes, u.cell_data.target_volumes, u.cell_data.target_volumes_float, u.cell_data.target_surface_areas, u.cell_data.surface_areas, u.cell_data.free_surfaces, u.cell_data.inhibition_states, u.N_cells, evt.alpha, evt.beta, evt.gamma, evt.dt)
CorePotts.get_event_ndrange(::ApplyMonolayerGrowthEvent, mask, u) = length(u.cell_data.volumes)

struct MonolayerGrowthEvent <: CorePotts.AbstractMultiEvent
    alpha::Float32
    beta::Float32
    gamma::Float32
    dt::Float32
    max_cells::Int
end

CorePotts.get_sub_events(evt::MonolayerGrowthEvent) = (
    ResetFreeSurfacesEvent(),
    AccumulateFreeSurfacesEvent(),
    ApplyMonolayerGrowthEvent(evt.alpha, evt.beta, evt.gamma, evt.dt)
)


In [ ]:
println("Setting up the Monolayer Benchmark Model on GPU...")

grid_size_main = (1200, 1200)
max_cells = 12_000

import Random; Random.seed!(42)
X1 = 2.0f0 + 0.4f0 * randn(Float32)
initial_div_thresh = floor(Int32, initial_area * X1)

sys = PottsSystem(
    cell_types = [medium, tissue],
    penalties = [
        HSTVolumeComponent(tissue => (λ = 5.0f0, target = 50.0f0);eta = 1.0f0),
        # Moore Topology
        SurfaceAreaComponent(tissue => (λ = 2.0f0, target = 12.0f0*sqrt(initial_area))),
        AdhesionComponent(
            (medium, tissue) => 10.0f0, 
            (tissue, tissue) => 20.0f0
        ),
        # ConnectivityConstraint(),
        MonolayerGrowthComponent(initial_area, initial_div_thresh)
    ],
    check_interval = 1,
    events = (
        MitosisEvent(tissue,
            trigger = CustomTrigger(monolayer_mitosis_trigger),
            inheritance = (
                target_volumes = CorePotts.Split(0.5f0),
                target_volumes_float = CorePotts.Split(0.5f0),
                division_threshold_volumes = CorePotts.Split(0.5f0),
            ),
            action = monolayer_post_mitosis_action
        ),
        MonolayerGrowthEvent(calibrated_alpha, 0.8f0, 0.2f0, 1.0f0, max_cells)
    )
)

layout = HypersphereLayout(tissue, (600, 600), Int(round(R)))

prob = PottsProblem(sys, layout, grid_size_main; 
                    max_cells = max_cells, 
                    tspan=(0, 20_000),
                    topology = CorePotts.NoFluxMooreTopology{2}())

import Adapt
prob = Adapt.adapt(ArrayType, prob)
alg_main = CheckerboardMetropolis(T=20.0f0, active_fraction=1.0f0)

println("Starting Simulation until 10,000 cells...")

# Check the cell count every 10.0 units of simulation time
terminate_cb = SciMLBase.PeriodicCallback(
    (integrator) -> begin
        # 100% Zero allocations, and the GPU only halts occasionally!
        if Metal.@allowscalar(integrator.u.N_cells[1]) >= 10000
            terminate!(integrator)
        end
    end, 
    10.0 # The exact time interval to check at
)

@time sol = CommonSolve.solve(prob, alg_main; callback = terminate_cb, saveat = 50)
Metal.@sync backend
println("Simulation Complete! Final cell count: ", sol.u[end].N_cells[])




In [ ]:
# --- NATIVE JULIA CONCAVEMAN PORT ---
function sq_dist(p1, p2)
    dx = p1[1] - p2[1]
    dy = p1[2] - p2[2]
    return dx*dx + dy*dy
end

function sq_seg_dist(p, p1, p2)
    x = p1[1]
    y = p1[2]
    dx = p2[1] - x
    dy = p2[2] - y
    if dx != 0 || dy != 0
        t = ((p[1] - x) * dx + (p[2] - y) * dy) / (dx*dx + dy*dy)
        if t > 1
            x = p2[1]
            y = p2[2]
        elseif t > 0
            x += dx * t
            y += dy * t
        end
    end
    dx = p[1] - x
    dy = p[2] - y
    return dx*dx + dy*dy
end

function orient2d(ax, ay, bx, by, cx, cy)
    return (ay - cy) * (bx - cx) - (ax - cx) * (by - cy)
end

function cross(p1, p2, p3)
    return orient2d(p1[1], p1[2], p2[1], p2[2], p3[1], p3[2])
end

function intersects(p1, q1, p2, q2)
    return p1 != q2 && q1 != p2 &&
           (cross(p1, q1, p2) > 0) != (cross(p1, q1, q2) > 0) &&
           (cross(p2, q2, p1) > 0) != (cross(p2, q2, q1) > 0)
end

function point_in_polygon(point, vs)
    x = point[1]
    y = point[2]
    inside = false
    j = length(vs)
    for i in 1:length(vs)
        xi = vs[i][1]
        yi = vs[i][2]
        xj = vs[j][1]
        yj = vs[j][2]
        intersect = ((yi > y) != (yj > y)) && (x < (xj - xi) * (y - yi) / (yj - yi) + xi)
        if intersect
            inside = !inside
        end
        j = i
    end
    return inside
end

function fast_convex_hull(points)
    left = points[1]
    top = points[1]
    right = points[1]
    bottom = points[1]
    for p in points
        if p[1] < left[1]; left = p; end
        if p[1] > right[1]; right = p; end
        if p[2] < top[2]; top = p; end
        if p[2] > bottom[2]; bottom = p; end
    end
    cull = [left, top, right, bottom]
    filtered = copy(cull)
    for p in points
        if !point_in_polygon(p, cull)
            push!(filtered, p)
        end
    end
    sort!(filtered, by = p -> (p[1], p[2]))
    lower = typeof(points)()
    for i in 1:length(filtered)
        while length(lower) >= 2 && cross(lower[end-1], lower[end], filtered[i]) <= 0
            pop!(lower)
        end
        push!(lower, filtered[i])
    end
    upper = typeof(points)()
    for i in length(filtered):-1:1
        while length(upper) >= 2 && cross(upper[end-1], upper[end], filtered[i]) <= 0
            pop!(upper)
        end
        push!(upper, filtered[i])
    end
    pop!(upper)
    pop!(lower)
    return [lower; upper]
end

mutable struct Node
    p::Vector{Float64}
    prev::Union{Node, Nothing}
    next::Union{Node, Nothing}
    function Node(p)
        new(p, nothing, nothing)
    end
end

function insert_node!(p, prev_node)
    node = Node(p)
    if prev_node === nothing
        node.prev = node
        node.next = node
    else
        node.next = prev_node.next
        node.prev = prev_node
        prev_node.next.prev = node
        prev_node.next = node
    end
    return node
end

function no_intersections(a, b, edges)
    for e in edges
        if intersects(e.p, e.next.p, a, b)
            return false
        end
    end
    return true
end

function find_candidate(points, a, b, c, d, max_dist, edges)
    candidates = []
    for p in points
        dist = sq_seg_dist(p, b, c)
        if dist <= max_dist
            push!(candidates, (dist, p))
        end
    end
    sort!(candidates, by = x -> x[1])
    for (dist, p) in candidates
        d0 = sq_seg_dist(p, a, b)
        d1 = sq_seg_dist(p, c, d)
        if dist < d0 && dist < d1 && no_intersections(b, p, edges) && no_intersections(c, p, edges)
            return p
        end
    end
    return nothing
end

function concaveman(points, concavity=1.5, lengthThreshold=0.0)
    concavity = max(0.0, concavity)
    sqConcavity = concavity * concavity
    sqLenThreshold = lengthThreshold * lengthThreshold
    hull_pts = fast_convex_hull(points)
    hull_set = Set(hull_pts)
    inner_points = [p for p in points if !(p in hull_set)]
    queue = Node[]
    edges = Node[]
    local last_node = nothing
    for p in hull_pts
        last_node = insert_node!(p, last_node)
        push!(queue, last_node)
        push!(edges, last_node)
    end
    while !isempty(queue)
        node = popfirst!(queue)
        a = node.p
        b = node.next.p
        sqLen = sq_dist(a, b)
        if sqLen < sqLenThreshold
            continue
        end
        maxSqLen = sqLen / sqConcavity
        c_node = node.prev.p
        d_node = node.next.next.p
        p = find_candidate(inner_points, c_node, a, b, d_node, maxSqLen, edges)
        if p !== nothing && min(sq_dist(p, a), sq_dist(p, b)) <= maxSqLen
            push!(queue, node)
            new_node = insert_node!(p, node)
            push!(queue, new_node)
            filter!(x -> x != p, inner_points)
            push!(edges, new_node)
        end
    end
    concave = typeof(points)()
    curr = last_node
    while true
        push!(concave, curr.p)
        curr = curr.next
        if curr === last_node
            break
        end
    end
    push!(concave, curr.p)
    return concave
end


In [ ]:
import Statistics: mean, std

function get_centroids(u)
    grid_cpu = Array(u.grid)
    n_cells = (u.N_cells isa Base.RefValue ? u.N_cells[] : Int(Array(u.N_cells)[1]))
    centroids_x = zeros(Float64, n_cells)
    centroids_y = zeros(Float64, n_cells)
    counts = zeros(Int, n_cells)
    for x in 1:size(grid_cpu, 1), y in 1:size(grid_cpu, 2)
        cid = grid_cpu[x,y]
        if cid > 0 && cid <= n_cells
            centroids_x[cid] += x
            centroids_y[cid] += y
            counts[cid] += 1
        end
    end
    pts = Vector{Vector{Float64}}()
    for i in 1:n_cells
        if counts[i] > 0
            push!(pts, [centroids_x[i] / counts[i] / R, centroids_y[i] / counts[i] / R])
        end
    end
    return pts
end

function get_concaveman_stats(u)
    pts = get_centroids(u)
    if length(pts) <= 3
        return (area=0.0, r_mean=0.0, width=0.0, roughness=1.0)
    end
    hull = concaveman(pts, 1.5, 0.0)
    area = 0.0
    C = 0.0
    for j in 1:length(hull)-1
        area += hull[j][1]*hull[j+1][2] - hull[j+1][1]*hull[j][2]
        C += sqrt((hull[j][1]-hull[j+1][1])^2 + (hull[j][2]-hull[j+1][2])^2)
    end
    area = abs(area) / 2.0
    cx = sum(p[1] for p in hull[1:end-1]) / (length(hull)-1)
    cy = sum(p[2] for p in hull[1:end-1]) / (length(hull)-1)
    dist_from_c = [sqrt((p[1]-cx)^2 + (p[2]-cy)^2) for p in hull[1:end-1]]
    r_mean = mean(dist_from_c)
    width = std(dist_from_c)
    C_circle = 2.0 * sqrt(pi * area)
    roughness = C / C_circle
    return (area=area, r_mean=r_mean, width=width, roughness=roughness)
end

openvt_metrics = [
    "Number of cells N" => (u) -> Float32((u.N_cells isa Base.RefValue ? u.N_cells[] : Int(Array(u.N_cells)[1]))),
    "Tissue radius r [R]" => (u) -> Float32(get_concaveman_stats(u).r_mean),
    "Tissue area a [R²]" => (u) -> Float32(get_concaveman_stats(u).area),
    "Boundary width w [R]" => (u) -> Float32(get_concaveman_stats(u).width),
    "Boundary roughness C/C_circle" => (u) -> Float32(get_concaveman_stats(u).roughness),
    "Growing cell fraction" => (u) -> begin
        n_cells = (u.N_cells isa Base.RefValue ? u.N_cells[] : Int(Array(u.N_cells)[1]))
        n_cells == 0 && return 0.0f0
        inhib = Array(u.cell_data.inhibition_states)[1:n_cells]
        return Float32(count(==(0), inhib) / n_cells)
    end
]


In [ ]:
using CairoMakie
CairoMakie.activate!()
using MakiePotts
using Colors

# Custom color scheme mapping inhibition states
color_map = Dict(
    0 => colorant"#50C878", # Green: Growing
    1 => colorant"#FF6B6B", # Red: Pressure Inhibited
    2 => colorant"#4A90E2", # Blue: Contact Inhibited
    3 => colorant"#333333"  # Black: Both
)

cell_color_func = (t) -> [color_map[Int(st)] for st in Array(sol.u[t].cell_data.inhibition_states)[1:(sol.u[t].N_cells isa Base.RefValue ? sol.u[t].N_cells[] : Int(Array(sol.u[t].N_cells)[1]))]]

# Index 1 = Medium (Dark), Index 2 = State 0 (Green), etc.
my_colors = ["#0B0F19", "#00E5FF", "#FF0055", "#7000FF", "#3A0088"]

record_potts("openvt_dashboard.mp4", sol;
    metrics = openvt_metrics,
    resolution = (1600, 1000),
    framerate = 20,
    color_property = :inhibition_states, 
    color_offset = 2,
    type_colors = my_colors,
    draw_boundaries = false,           # <--- Turns on the cell outlines!
    boundary_color = "#000000"        # (Optional) Black boundaries are the default
)